In [ ]:
# Configuración de GPU
import torch

# Validar si esta disponible una GPU
print('GPU disponible:', torch.cuda.is_available())

# Seleccionar GPU si esta disponible.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

GPU disponible: True


In [ ]:
# Importación de librerias necesarias
import gc
import time
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import model_info
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

#%pip -q install "transformers>=4.44,<5" "datasets>=3,<5" "scikit-learn>=1.4,<2" "huggingface_hub>=0.25,<2" "pandas>=2,<3" "matplotlib>=3.8,<4" pillow


In [ ]:
# Autenticación en Hugging Face Hub.
# Dado que se utilizarán modelos públicos no es neceario realizar el login,
# de acuerdo a la rubrica se añade el proceso para el inicio de sesión

# Importa la función para iniciar sesión en Hugging Face.
from huggingface_hub import login
from getpass import getpass

# Bandera para utilizar login de Hugging Face
login = False
if login:
    login(token=getpass('Token de Hugging Face: '))
else:
    print('No se ha iniciado sesión en Huggin Face')

No se ha iniciado sesión en Huggin Face


In [ ]:
#Carga de modelos y datasets

#Modelos a utilizar
MODELS = ['openai/clip-vit-base-patch32', 'openai/clip-vit-base-patch16', 'openai/clip-vit-large-patch14']

#Etiquetas del dataset
LABELS = ['angular_leaf_spot', 'bean_rust', 'healthy']

#Prompts base para clasificar las imagenes
PROMPTS = ['a photo of a bean leaf with angular leaf spot disease', 'a photo of a bean leaf with bean rust disease', 'a photo of a healthy bean leaf']

#Muestra de imagenes para probar
N = 60

#Semillla para la reproducibilidad
SEED = 42

#Carga del dataset
raw = load_dataset('AI-Lab-Makerere/beans', split='test')

#Información del dataset
print('Campos:', raw.features)
print('Imágenes de prueba disponibles:', len(raw))
names = raw.features['labels'].names
print('Clases:', names)

#Mezclado de la muestra
raw = raw.shuffle(seed=SEED).select(range(min(N, len(raw))))

#Conversión de etiquetas a numero
y_true = [LABELS.index(names[int(x)]) for x in raw['labels']]

README.md:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  144MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.5MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

Campos: {'image_file_path': Value('string'), 'image': Image(mode=None, decode=True), 'labels': ClassLabel(names=['angular_leaf_spot', 'bean_rust', 'healthy'])}
Imágenes de prueba disponibles: 128
Clases: ['angular_leaf_spot', 'bean_rust', 'healthy']


In [ ]:
#Pipeline de inferencia

#Función para evaluar un modelo y devolver sus resultados
def evaluate(model_id):
    print('Modelo:', model_id)
    #Descarga el procesador de imágenes y textos
    processor = CLIPProcessor.from_pretrained(model_id)
    #Cargar el modelo en modo evaluación
    model = CLIPModel.from_pretrained(model_id).to(DEVICE).eval()
    #Conviersión de las descripciones en tensores
    text_inputs = processor(text=PROMPTS, return_tensors='pt', padding=True).to(DEVICE)

    predictions, times = [], []
    with torch.inference_mode():
        text_features = model.get_text_features(**text_inputs)
        if hasattr(text_features, 'pooler_output'):
            text_features = text_features.pooler_output
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        for i, item in enumerate(raw):
            #Conviersión de una imagen RGB a tensores.
            inputs = processor(images=item['image'].convert('RGB'), return_tensors='pt').to(DEVICE)
            if i == 0:
                _ = model.get_image_features(**inputs)
                if DEVICE == 'cuda':
                    torch.cuda.synchronize()
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            img_features = model.get_image_features(**inputs)
            if hasattr(img_features, 'pooler_output'):
                img_features = img_features.pooler_output
            if DEVICE == 'cuda':
                torch.cuda.synchronize()
            #Calculo de la latencia en milisegundos.
            elapsed = (time.perf_counter() - t0) * 1000
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)
            logits = img_features @ text_features.T
            predictions.append(int(logits.argmax(dim=-1).item()))
            times.append(elapsed)
    #Cálculo de métricas (accuracy, precision, recall, F1-score).
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, predictions, average='macro', zero_division=0)

    #Obtención de los detalles de cada modelo
    row = dict(
        modelo=model_id,
        parametros_millones=round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
        accuracy=accuracy_score(y_true, predictions),
        precision=pr,
        recall=rc,
        f1=f1,
        latencia_ms_mediana=float(np.median(times)),
        latencia_ms_p95=float(np.percentile(times, 95)),
        dispositivo=DEVICE,
        imagenes=len(raw),
    )

    # Impresión de metricas
    print(classification_report(y_true, predictions, labels=list(range(len(LABELS))), target_names=LABELS, zero_division=0))
    del model, processor, text_inputs, text_features
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return row
#Ejecución de los modelos
results = [evaluate(mid) for mid in MODELS]

Modelo: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

                   precision    recall  f1-score   support

angular_leaf_spot       0.33      0.95      0.49        19
        bean_rust       0.80      0.19      0.31        21
          healthy       0.00      0.00      0.00        20

         accuracy                           0.37        60
        macro avg       0.38      0.38      0.26        60
     weighted avg       0.38      0.37      0.26        60



In [ ]:
#Tabla de resultados
df = pd.DataFrame(results).sort_values('f1', ascending=False)  # Crea y ordena la tabla por F1 macro.
display(df.round(4))  # Muestra los resultados con cuatro decimales.

,modelo,parametros_millones,accuracy,precision,recall,f1,latencia_ms_mediana,latencia_ms_p95,dispositivo,imagenes
2,openai/clip-vit-large-patch14,427.6,0.3667,0.3758,0.3793,0.2647,55.1093,56.0321,cuda,60
0,openai/clip-vit-base-patch32,151.3,0.3833,0.2035,0.3684,0.2444,11.1183,23.2714,cuda,60
1,openai/clip-vit-base-patch16,149.6,0.3333,0.1818,0.3191,0.2309,34.5210,61.0552,cuda,60


## Conclusiones parciales

De acuerdo a la tabla se resultados se puede concluir que el mejor modelo de acuerdo a la mejor puntuación F1 es openai/clip-vit-large-patch14, el mejor modelo de acuerdo a la latencia es el openai/clip-vit-base-patch32, todos los modelos tienen una precisión de aproximadamente el 33% esto es debido a que se esta utilizando el modelo sin realizar un entrenamiento con el dataset muestra